# Random Forest — Model Development

## Project Overview
This notebook demonstrates an end-to-end implementation of a Random Forest model to predict the target variable based on the given input features. The project covers data preprocessing, model training, evaluation, and model serialization for deployment.

## Business Problem
Many real-world business problems require predicting continuous values accurately, even when the relationship between input features and the target is non-linear. This project builds a Random Forest model that learns these relationships and provides reliable predictions for unseen data.

## Workflow
1. Import required libraries
2. Load and inspect the dataset
3. Perform data preprocessing
4. Encode categorical features (if applicable)
5. Split the dataset into training and testing sets
6. Standardize the features
7. Train the Random Forest model
8. Evaluate model performance using R² Score
9. Persist the trained model using Pickle

## Expected Outcome
A production-ready serialized model (`final_random_forest_model.sav`) that can be deployed for making predictions on new data.

In [1]:
# Import library.
import pandas as pd

In [2]:
# Read the input dataset.
dataset = pd.read_csv('../data/50_Startups.csv')

In [3]:
# Load the first five rows of dataset.
dataset.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [4]:
# Convert the nominal data in the state column by one-hot encoding.
dataset = pd.get_dummies(dataset, drop_first=True).astype(int)

In [5]:
# Load the first five rows of dataset after one-hot encoding.
dataset.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_Florida,State_New York
0,165349,136897,471784,192261,0,1
1,162597,151377,443898,191792,0,0
2,153441,101145,407934,191050,1,0
3,144372,118671,383199,182901,0,1
4,142107,91391,366168,166187,1,0


In [6]:
# Assign columns to independent variable.
independent = dataset[['R&D Spend',	'Administration', 'Marketing Spend', 'State_Florida', 'State_New York']]

In [7]:
# Assign columns to dependent variable.
dependent = dataset[['Profit']]

In [8]:
# Split the training set and test set from the dataset (input).
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(independent, dependent, test_size=0.3, random_state=0)

In [9]:
# Create an instance of RandomForestRegressor.
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Crete a Random Forest model.
model = RandomForestRegressor(random_state=0)

# Define parameter values
param_grid = {
    "criterion": ["squared_error", "absolute_error", "poisson"],
    "n_estimators": range(10, 50, 2)
}

# Create a GridSearchCV object to find the best combination of hyperparameters
grid = GridSearchCV(
    estimator=model,          # Machine Learning model to be tuned
    param_grid=param_grid,    # Dictionary containing hyperparameter values to test
    cv=5,                     # Perform 5-fold cross-validation
    scoring="r2",             # Use R² score to evaluate each parameter combination
    n_jobs=-1                 # Use all available CPU cores for faster execution
)

# Train the model using every hyperparameter combination and perform cross-validation
grid.fit(X_train, np.ravel(y_train))

# Display the hyperparameter combination that achieved the highest average R² score
print("Best Parameters:")
print(grid.best_params_)

# Display the best average cross-validation R² score
print("\nBest Score:")
print(grid.best_score_)

# Retrieve the model trained with the best hyperparameters
best_model = grid.best_estimator_

Best Parameters:
{'criterion': 'poisson', 'n_estimators': 14}

Best Score:
0.8833232701035258


In [10]:
# Convert all GridSearchCV results into a Pandas DataFrame
results = pd.DataFrame(grid.cv_results_)

# Select only the columns required for analysis
results = results[
    [
        "param_criterion",      # Splitting criterion used
        "param_n_estimators",       # Splitting strategy used
        "mean_test_score",      # Average R² score across all cross-validation folds
        "rank_test_score"       # Ranking of each parameter combination (1 = Best)
    ]
]

# Rename the column names to make the output more readable
results.rename(columns={
    "param_criterion": "Criterion",
    "param_n_estimators": "n_estimators",
    "mean_test_score": "Mean R2 Score",
    "rank_test_score": "Rank"
}, inplace=True)

# Sort the results by rank so that the best-performing parameter combination appears first
results = results.sort_values(by="Rank", ascending=True)

# Save the GridSearchCV results to an Excel file
results.to_excel("../outputs/RF_GridSearch_Results.xlsx", index=False)

# Display the final sorted results
print(results)

         Criterion  n_estimators  Mean R2 Score  Rank
42         poisson            14       0.883323     1
22  absolute_error            14       0.881711     2
21  absolute_error            12       0.879236     3
41         poisson            12       0.876728     4
40         poisson            10       0.876607     5
25  absolute_error            20       0.876556     6
26  absolute_error            22       0.875525     7
23  absolute_error            16       0.874993     8
2    squared_error            14       0.874430     9
46         poisson            22       0.874363    10
43         poisson            16       0.874187    11
45         poisson            20       0.873142    12
44         poisson            18       0.873031    13
24  absolute_error            18       0.872890    14
1    squared_error            12       0.872853    15
37  absolute_error            44       0.872619    16
31  absolute_error            32       0.872153    17
30  absolute_error          

In [11]:
# Save the model to pickle library.
import pickle

# Save the best trained model to a pickle file
with open("../models/final_random_forest_model.sav", "wb") as file:
    pickle.dump(best_model, file)

print("Model saved successfully!")

Model saved successfully!
